**Подключение библиотек** <span style="color:#FF0000">(Тебе надо)</span>

In [ ]:
import numpy as np                    # Математика
import pandas as pd                   # Таблицы
import scipy                          # Математика+
from scipy.optimize import curve_fit  # Аппроксимация
import matplotlib.pyplot as plt       # Графики
import cv2                            # Компьютерное зрение
from PIL import Image                 # Картинки

**Чтение таблицы**
- Загружается Excel с показаниями динамометра
- В конце раздела показания динамометра в numpy-векторе `force` 

#### <span style="color:#FF0000">Тебе надо:</span>
- Указать название Excel-таблицы

In [ ]:
data = pd.read_excel("storage/Браес/експеримент 7 21.11.2025.xlsx",
                     sheet_name="Svodnaia tablitca")
data 

In [ ]:
# Последовательность усилия
force = data["Датчик усилия, Н"].to_numpy()

plt.plot(force)
plt.xlabel("Время")
plt.ylabel("Датчик усилия, Н")
plt.grid()
plt.show()

**Калибровка по видео**
- Загружается видео с шахматной доской
- В конце раздела 1) коэффициенты дисторсии `dist_coeffs` 2) матрица А с параметрами камеры `camera_matrix`


$$A=\begin{bmatrix}   f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1      \end{bmatrix}$$

#### <span style="color:#FF0000">Тебе надо:</span>
- Указать название видео
- Указать удачные кадры
    - Сначала угадай номер кадра $N_{кадра}\approx t_{сек}*25 \frac{кадров}{сек}$
    - Потом посмотри на выбранные кадры в "проверка выбранных кадров" (ниче там не меняй!)
    - Выбери ~30 хороших кадров с разных ракурсов доски
    - Можешь посмотреть как проходит исправление искажения. Если оно +- симметричное то окей. Если несимметричное $-$ калибровка говно.
- Проверить размер шахматной доски

In [ ]:
# Загрузка видео  
cap = cv2.VideoCapture("storage/Браес/DSC_0021.MOV")  

# Кадры, которые нам нравятся
frames_id = [185, 190, 192, 201, 675, 679]  

# Размер шахматной доски
chessboard_size = 9, 7       # Количество ячеек
square_size = 20             # Размер 1 ячейки в миллиметрах

↓ проверка выбранных кадров ↓

In [ ]:
counter_frame, counter_figure, frames = 0, 0, []
nrow = int(np.ceil(len(frames_id)/3))
fig, ax = plt.subplots(nrow, 3, figsize=(16,9*nrow/3))

while True:  # Перебор кадров
    counter_frame += 1
    add_info, frame = cap.read()                     # Чтение кадра 
    if not add_info:                                 # Если кадры в видео закончились - остановить
        break 
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)   # Цветное изоб-е в чёрно-белое
    if counter_frame in frames_id:
        frames.append(gray)
        ax[int(counter_figure//3)][counter_figure % 3].imshow(Image.fromarray(gray), cmap='gray')
        counter_figure += 1
        if counter_figure == len(frames_id):         # Если кадры, которые нам нравятся, закончились - остановить
            break
plt.show()

↓ калибровка камеры ↓

In [ ]:
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
objp = np.zeros((chessboard_size[0] * chessboard_size[1],3), np.float32)
objp[:,:2] = np.mgrid[0:chessboard_size[0],0:chessboard_size[1]].T.reshape(-1,2)
objp = objp * square_size
objpoints = [] # 3d point in real world space
imgpoints = [] # 2d points in image plane.

for gray in frames: 
    # Поиск углов шахматной доски
    ret, corners = cv2.findChessboardCorners(gray, chessboard_size, None)
 
    # If found, add object points, image points (after refining them)
    if ret == True:
        objpoints.append(objp)
        corners2 = cv2.cornerSubPix(gray,corners, (11,11), (-1,-1), criteria)
        imgpoints.append(corners2)

# Калибровка камеры
ret, mtx, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)
h,  w = gray.shape
A, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w,h), 1, (w,h))

print(f"Размер изображения: [{h},{w}]")
print(f"Фокусные расстояния Ох, Оу: {A[0,0]:.2f}, {A[1,1]:.2f}")
print(f"Точка центра изображения: [{A[0,2]:.2f}, {A[1,2]:.2f}]")
print()
print(f"# Коэффициенты дисторсии: \ndist_coeffs=np.array([{[float(i) for i in dist_coeffs[0]]}])")
print(f"# Матрица А: \ncamera_matrix=np.array([\n", end="")
for line in A:
    print(f"{[float(i) for i in line]},")
print(f"])")

↓ посмотреть как прошла калибровка: исправление искажения изображения ↓

In [ ]:
fig, ax = plt.subplots(len(frames), 2, figsize=(16,9*len(frames)/2))
for i, gray in enumerate(frames): 
    dst = cv2.undistort(gray, mtx, dist, None, newcameramtx)
    ax[i][0].imshow(Image.fromarray(gray), cmap='gray')
    ax[i][1].imshow(Image.fromarray(dst), cmap='gray')
plt.show()

**Проверка того, что метка действительно есть в словаре**

In [ ]:
N = 0  # Номер метки

aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
img = cv2.aruco.generateImageMarker(aruco_dict, N, 100)
plt.imshow(img, cmap = 'gray', interpolation = "nearest")
plt.title(f"N = {N} / 50")
plt.axis("off")
plt.show()

**Чтение Aruco-меток**

#### <span style="color:#FF0000">Тебе надо:</span>
- Скопировать вниз с вывода "калибровка камеры" коэффициенты дисторсии, матрицу А
- Установить размер распечатанной метки в миллиметрах
- Указать название видео
- Обработать видео
    - Цикл `while` начинает проверять кадры с последнего. Перед каждой обработкой видео запускай предыдущую ячейку Jupyter c функцией `cv2.VideoCapture`
    - Сейчас цикл `while` сбрасывается при первой найденной метке. Если вдруг у тебя метка вообще находится, удалить блок, обозначенный "Удалить потом"

In [ ]:
# Размер метки в мм
marker_size = 30

# Путь к видео
cap = cv2.VideoCapture("storage/Браес/DSC_0008.MOV")  

# СКОПИРУЙ СВЕРХУ ↓

# Коэффициенты дисторсии: 
dist_coeffs=np.array([[0.4047110977896863, -2.9459530367274027, 0.007295497366415143, -0.051497355660016786, 10.801561239790276]])
# Матрица А: 
camera_matrix=np.array([
[4031.7575964345983, 0.0, 497.0247846735959],
[0.0, 4084.5370243540574, 811.287187631327],
[0.0, 0.0, 1.0],
])

↓ обработка видео ↓

In [ ]:
# Выбор словаря Aruco: 4X4 - кол-во квадратиков в метке, 50 - кол-во меток в словаре (метки разные в разных словарях)
dictionary = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)  
detectorParams = cv2.aruco.DetectorParameters()
detector = cv2.aruco.ArucoDetector(dictionary, detectorParams)   # Инициализация детектора

counter = 0
docking = []
while True:
    counter += 1
    add_info, frame = cap.read()                                     # Чтение кадра 
    if not add_info:                                                 # Если кадры закончились - остановить
        break 
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)                   # Цветное изоб-е в чёрно-белое
    corners, ids, rejected = detector.detectMarkers(gray)            # Поиск маркеров Aruco
    
    if ids is None:
        print(f"Кадр {counter} | Метка не обнаружена")
    else:
        for i, id_ in enumerate(ids):
            # Оценка положения маркера
            rvecs, tvecs, _ = cv2.aruco.estimatePoseSingleMarkers(corners, marker_size, camera_matrix, dist_coeffs)
            print(f"Кадр {counter} | Нашлась метка id={id_}, положение={tvecs[i][0]}, поворот={rvecs[i][0]}")
    
            ########## Удалить потом
            im = Image.fromarray(gray)
            plt.figure(figsize=(16/2,9/2))
            plt.imshow(im, cmap='gray')
            plt.show()
            break 
            ########## Удалить потом
            
            # Запись параметров
            docking.append([counter, id_, tvecs[i], rvecs[i]])

↓ сохранение траектории меток в файл ↓